# VEEV Variant Analysis

Analyze annotated variants from both iVar and LoFreq variant callers across different time points (1, 3, 5 DPI).

## Data Overview
- **LoFreq**: Annotated VCF files from LoFreq variant calling
- **iVar**: Annotated VCF files converted from iVar TSV format
- **Time points**: 1, 3, 5 days post-infection (DPI)
- **Replicates**: 4 biological replicates per time point

In [36]:
# Import required libraries
import pandas as pd
import numpy as np
import glob
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pysam

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


Define data loading functions

In [37]:
def parse_bcsq(bcsq_field):
    """
    Parse BCSQ annotation field into individual components.
    BCSQ format: consequence|gene|transcript|biotype|strand|amino_acid_change|dna_change
    """
    if bcsq_field is None:
        return {
            'consequence': None,
            'gene': None,
            'transcript': None,
            'biotype': None,
            'strand': None,
            'amino_acid_change': None,
            'dna_change': None
        }
    
    # Handle both tuple format (LoFreq) and string format (iVar)
    bcsq_str = bcsq_field
    if isinstance(bcsq_field, tuple):
        if len(bcsq_field) > 0:
            bcsq_str = bcsq_field[0]
        else:
            bcsq_str = None
    
    if bcsq_str is None or bcsq_str == '' or str(bcsq_str).startswith('@'):
        return {
            'consequence': None,
            'gene': None,
            'transcript': None,
            'biotype': None,
            'strand': None,
            'amino_acid_change': None,
            'dna_change': None
        }
    
    # Split by pipe
    parts = str(bcsq_str).split('|')
    
    return {
        'consequence': parts[0] if len(parts) > 0 else None,
        'gene': parts[1] if len(parts) > 1 else None,
        'transcript': parts[2] if len(parts) > 2 else None,
        'biotype': parts[3] if len(parts) > 3 else None,
        'strand': parts[4] if len(parts) > 4 else None,
        'amino_acid_change': parts[5] if len(parts) > 5 else None,
        'dna_change': parts[6] if len(parts) > 6 else None
    }

def parse_vcf_file(vcf_path):
    """
    Parse VCF file using pysam with cleaner column names and parsed BCSQ.
    Handle problematic VCF headers more robustly.
    """
    try:
        variants = []
        vcf_in = pysam.VariantFile(vcf_path)
        
        for record in vcf_in:
            # Extract ALT allele properly
            alt_allele = '.'
            if record.alts:
                alt_allele = str(record.alts[0])
                
            # Extract INFO fields with proper handling of missing values
            variant_data = {
                'CHROM': record.chrom,
                'POS': record.pos,
                'REF': record.ref,
                'ALT': alt_allele,
                'QUAL': record.qual if record.qual is not None else '.',
                'DP': record.info.get('DP', None),
                'AF': record.info.get('AF', None), 
                'SB': record.info.get('SB', None),
                'DP4': record.info.get('DP4', None),
                'BCSQ_raw': record.info.get('BCSQ', None)
            }
            
            # Parse BCSQ into separate columns
            bcsq_parsed = parse_bcsq(variant_data['BCSQ_raw'])
            variant_data.update(bcsq_parsed)
            
            variants.append(variant_data)
        
        vcf_in.close()
        return pd.DataFrame(variants) if variants else pd.DataFrame()
        
    except Exception as e:
        print(f"Error parsing {vcf_path}: {str(e)}")
        # Try alternative approach for problematic files
        try:
            return parse_vcf_manually(vcf_path)
        except:
            return pd.DataFrame()

def parse_vcf_manually(vcf_path):
    """
    Fallback manual parsing for problematic VCF files.
    NOW INCLUDES iVar frequency extraction from FORMAT field.
    """
    variants = []
    with open(vcf_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('#') or not line:
                continue
            
            parts = line.split('\t')
            if len(parts) >= 8:
                variant_data = {
                    'CHROM': parts[0],
                    'POS': int(parts[1]),
                    'REF': parts[3],
                    'ALT': parts[4],
                    'QUAL': parts[5] if parts[5] != '.' else None,
                    'DP': None,
                    'AF': None,
                    'SB': None,
                    'DP4': None,
                    'BCSQ_raw': None
                }
                
                # Parse INFO field
                info_parts = parts[7].split(';')
                for info in info_parts:
                    if '=' in info:
                        key, value = info.split('=', 1)
                        if key == 'DP':
                            try:
                                variant_data['DP'] = int(value)
                            except:
                                pass
                        elif key == 'BCSQ':
                            variant_data['BCSQ_raw'] = value
                
                # Parse FORMAT field for iVar ALT_FREQ (if present)
                if len(parts) >= 10:  # Has FORMAT and sample columns
                    format_field = parts[8]
                    sample_data = parts[9]
                    format_keys = format_field.split(':')
                    sample_values = sample_data.split(':')
                    
                    # Extract ALT_FREQ if available
                    if 'ALT_FREQ' in format_keys and len(sample_values) > format_keys.index('ALT_FREQ'):
                        alt_freq_idx = format_keys.index('ALT_FREQ')
                        try:
                            variant_data['AF'] = float(sample_values[alt_freq_idx])
                        except:
                            variant_data['AF'] = None
                
                # Parse BCSQ into separate columns
                bcsq_parsed = parse_bcsq(variant_data['BCSQ_raw'])
                variant_data.update(bcsq_parsed)
                            
                variants.append(variant_data)
    
    return pd.DataFrame(variants) if variants else pd.DataFrame()

def extract_sample_info(filename):
    """Extract sample info from filename."""
    basename = os.path.splitext(os.path.basename(filename))[0]
    match = re.match(r'INH_(\d+)_DPI_R(\d+)_([A-Z]\d+)', basename)
    
    if match:
        return {
            'sample_id': basename,
            'dpi': int(match.group(1)),
            'replicate': int(match.group(2)),
            'plate': match.group(3)
        }
    return {'sample_id': basename, 'dpi': None, 'replicate': None, 'plate': None}

print("FIXED: Enhanced VCF parsing functions with proper iVar frequency extraction!")

FIXED: Enhanced VCF parsing functions with proper iVar frequency extraction!


Load Lofreq data

In [38]:
# Load LoFreq variants
print("Loading LoFreq variants...")
# Check current working directory


# Look for VCF files in the LoFreq subdirectory
lofreq_files = glob.glob("LoFreq/*.vcf")
if not lofreq_files:
    # If not found, try looking in parent directory structure
    lofreq_files = glob.glob("../Annotated_variants/LoFreq/*.vcf")
if not lofreq_files:
    # Try absolute path
    lofreq_files = glob.glob("/home/jonssonlab/Desktop/Alex/VEEV/044_NH/Annotated_variants/LoFreq/*.vcf")

print(f"Found {len(lofreq_files)} LoFreq VCF files")
if lofreq_files:
    print("Files found:")
    for f in lofreq_files[:3]:  # Show first 3 files
        print(f"  {f}")
    if len(lofreq_files) > 3:
        print(f"  ... and {len(lofreq_files) - 3} more")

lofreq_data = []
for vcf_file in lofreq_files:
    print(f"  Processing {os.path.basename(vcf_file)}...")
    df = parse_vcf_file(vcf_file)
    
    # Add sample information even for empty files
    sample_info = extract_sample_info(vcf_file)
    
    if not df.empty:
        # Add metadata for files with variants
        for key, value in sample_info.items():
            df[key] = value
        
        # Add variant caller info
        df['variant_caller'] = 'LoFreq'
        df['source_file'] = os.path.basename(vcf_file)
        
        lofreq_data.append(df)
    else:
        # Track empty files for summary
        print(f"    Empty file (no variants) - DPI {sample_info.get('dpi', 'unknown')}")

# Combine all LoFreq data
if lofreq_data:
    lofreq_df = pd.concat(lofreq_data, ignore_index=True)
    print(f"LoFreq DataFrame created with {len(lofreq_df)} variants from {len(lofreq_data)} samples with variants")
else:
    lofreq_df = pd.DataFrame()
    print("No LoFreq variants found")

print(f"LoFreq variants by DPI:")
if not lofreq_df.empty:
    print(lofreq_df.groupby('dpi').size())
else:
    print("No data to display")

Loading LoFreq variants...
Found 12 LoFreq VCF files
Files found:
  LoFreq/INH_5_DPI_R3_G3.vcf
  LoFreq/INH_3_DPI_R3_C3.vcf
  LoFreq/INH_1_DPI_R1_E2.vcf
  ... and 9 more
  Processing INH_5_DPI_R3_G3.vcf...
  Processing INH_3_DPI_R3_C3.vcf...
  Processing INH_1_DPI_R1_E2.vcf...
    Empty file (no variants) - DPI 1
  Processing INH_1_DPI_R4_H2.vcf...
    Empty file (no variants) - DPI 1
  Processing INH_3_DPI_R2_B3.vcf...
  Processing INH_1_DPI_R3_G3.vcf...
    Empty file (no variants) - DPI 1
  Processing INH_3_DPI_R4_D3.vcf...
  Processing INH_3_DPI_R1_A3.vcf...
  Processing INH_5_DPI_R2_F3.vcf...
  Processing INH_5_DPI_R1_E3.vcf...
  Processing INH_5_DPI_R4_H3.vcf...
  Processing INH_1_DPI_R2_F2.vcf...
    Empty file (no variants) - DPI 1
LoFreq DataFrame created with 3220 variants from 8 samples with variants
LoFreq variants by DPI:
dpi
3    1363
5    1857
dtype: int64


Load Ivar data

In [39]:
# Load iVar variants
print("Loading iVar variants...")

# Look for VCF files in the Ivar subdirectory
ivar_files = glob.glob("Ivar/*.vcf")
if not ivar_files:
    # If not found, try looking in parent directory structure
    ivar_files = glob.glob("../Annotated_variants/Ivar/*.vcf")
if not ivar_files:
    # Try absolute path
    ivar_files = glob.glob("/home/jonssonlab/Desktop/Alex/VEEV/044_NH/Annotated_variants/Ivar/*.vcf")

print(f"Found {len(ivar_files)} iVar VCF files")
if ivar_files:
    print("Files found:")
    for f in ivar_files[:3]:  # Show first 3 files
        print(f"  {f}")
    if len(ivar_files) > 3:
        print(f"  ... and {len(ivar_files) - 3} more")

ivar_data = []
for vcf_file in ivar_files:
    print(f"  Processing {os.path.basename(vcf_file)}...")
    df = parse_vcf_file(vcf_file)
    
    # Add sample information even for empty files
    sample_info = extract_sample_info(vcf_file)
    
    if not df.empty:
        # Add metadata for files with variants
        for key, value in sample_info.items():
            df[key] = value
        
        # Add variant caller info
        df['variant_caller'] = 'iVar'
        df['source_file'] = os.path.basename(vcf_file)
        
        ivar_data.append(df)
    else:
        # Track empty files for summary
        print(f"    Empty file (no variants) - DPI {sample_info.get('dpi', 'unknown')}")

# Combine all iVar data
if ivar_data:
    ivar_df = pd.concat(ivar_data, ignore_index=True)
    print(f"iVar DataFrame created with {len(ivar_df)} variants from {len(ivar_data)} samples with variants")
else:
    ivar_df = pd.DataFrame()
    print("No iVar variants found")

print(f"iVar variants by DPI:")
if not ivar_df.empty:
    print(ivar_df.groupby('dpi').size())
else:
    print("No data to display")

Loading iVar variants...
Found 12 iVar VCF files
Files found:
  Ivar/INH_5_DPI_R3_G3.vcf
  Ivar/INH_3_DPI_R3_C3.vcf
  Ivar/INH_1_DPI_R1_E2.vcf
  ... and 9 more
  Processing INH_5_DPI_R3_G3.vcf...
Error parsing Ivar/INH_5_DPI_R3_G3.vcf: Invalid header
  Processing INH_3_DPI_R3_C3.vcf...
Error parsing Ivar/INH_3_DPI_R3_C3.vcf: Invalid header
  Processing INH_1_DPI_R1_E2.vcf...
    Empty file (no variants) - DPI 1
  Processing INH_1_DPI_R4_H2.vcf...
    Empty file (no variants) - DPI 1
  Processing INH_3_DPI_R2_B3.vcf...
Error parsing Ivar/INH_3_DPI_R2_B3.vcf: Invalid header
  Processing INH_1_DPI_R3_G3.vcf...
    Empty file (no variants) - DPI 1
  Processing INH_3_DPI_R4_D3.vcf...
Error parsing Ivar/INH_3_DPI_R4_D3.vcf: Invalid header
  Processing INH_3_DPI_R1_A3.vcf...
Error parsing Ivar/INH_3_DPI_R1_A3.vcf: Invalid header
  Processing INH_5_DPI_R2_F3.vcf...
Error parsing Ivar/INH_5_DPI_R2_F3.vcf: Invalid header
  Processing INH_5_DPI_R1_E3.vcf...
Error parsing Ivar/INH_5_DPI_R1_E3.vcf:

Inspect data

In [40]:
# Summary of loaded data
print("="*60)
print("VARIANT LOADING SUMMARY")
print("="*60)

print(f"LoFreq variants: {len(lofreq_df) if not lofreq_df.empty else 0}")
print(f"iVar variants: {len(ivar_df) if not ivar_df.empty else 0}")
print(f"Total variants: {(len(lofreq_df) if not lofreq_df.empty else 0) + (len(ivar_df) if not ivar_df.empty else 0)}")

print("\nBreakdown by DPI and variant caller:")
if not lofreq_df.empty or not ivar_df.empty:
    # Combine for summary
    all_variants = []
    if not lofreq_df.empty:
        all_variants.append(lofreq_df[['dpi', 'variant_caller', 'sample_id']].copy())
    if not ivar_df.empty:
        all_variants.append(ivar_df[['dpi', 'variant_caller', 'sample_id']].copy())
    
    if all_variants:
        combined_summary = pd.concat(all_variants, ignore_index=True)
        summary_table = combined_summary.groupby(['dpi', 'variant_caller']).size().unstack(fill_value=0)
        print(summary_table)
        
        print("\nSamples processed:")
        sample_count = combined_summary.groupby(['dpi', 'variant_caller'])['sample_id'].nunique().unstack(fill_value=0)
        print(sample_count)
else:
    print("No variants loaded")

print("\n" + "="*60)

VARIANT LOADING SUMMARY
LoFreq variants: 3220
iVar variants: 3954
Total variants: 7174

Breakdown by DPI and variant caller:
variant_caller  LoFreq  iVar
dpi                         
3                 1363  1975
5                 1857  1979

Samples processed:
variant_caller  LoFreq  iVar
dpi                         
3                    4     4
5                    4     4



# Filter Variants

## Data Structure Overview

Now that we've loaded the data, let's examine the structure of our DataFrames to understand what information is available for analysis.

In [41]:
# Show lofreq df preview
lofreq_df.head()

,CHROM,POS,REF,ALT,QUAL,DP,AF,SB,DP4,BCSQ_raw,...,biotype,strand,amino_acid_change,dna_change,sample_id,dpi,replicate,plate,variant_caller,source_file
0,VEEV_INH,38,A,G,125.0,3473,0.003743,1,"(2545, 910, 12, 3)",None,...,None,None,None,None,INH_5_DPI_R3_G3,5,3,G3,LoFreq,INH_5_DPI_R3_G3.vcf
1,VEEV_INH,245,G,A,117.0,5228,0.003060,6,"(2154, 3053, 4, 12)",(synonymous|NSP1|NSP1_transcript|protein_codin...,...,protein_coding,+,67A,245G>A,INH_5_DPI_R3_G3,5,3,G3,LoFreq,INH_5_DPI_R3_G3.vcf
2,VEEV_INH,482,A,G,252.0,4879,0.004919,8,"(2358, 2491, 8, 16)",(synonymous|NSP1|NSP1_transcript|protein_codin...,...,protein_coding,+,146Q,482A>G,INH_5_DPI_R3_G3,5,3,G3,LoFreq,INH_5_DPI_R3_G3.vcf
3,VEEV_INH,611,T,C,65.0,4388,0.002051,1,"(2222, 2153, 4, 5)",(synonymous|NSP1|NSP1_transcript|protein_codin...,...,protein_coding,+,189A,611T>C,INH_5_DPI_R3_G3,5,3,G3,LoFreq,INH_5_DPI_R3_G3.vcf
4,VEEV_INH,687,T,C,79.0,4540,0.002423,22,"(2273, 2251, 1, 10)",(missense|NSP1|NSP1_transcript|protein_coding|...,...,protein_coding,+,215S>215P,687T>C,INH_5_DPI_R3_G3,5,3,G3,LoFreq,INH_5_DPI_R3_G3.vcf


In [42]:
# Show ivar df preview
ivar_df.head()

,CHROM,POS,REF,ALT,QUAL,DP,AF,SB,DP4,BCSQ_raw,...,biotype,strand,amino_acid_change,dna_change,sample_id,dpi,replicate,plate,variant_caller,source_file
0,VEEV_INH,15,A,G,None,1118,0.001789,None,None,None,...,None,None,None,None,INH_5_DPI_R3_G3,5,3,G3,iVar,INH_5_DPI_R3_G3.vcf
1,VEEV_INH,19,A,G,None,1441,0.001388,None,None,None,...,None,None,None,None,INH_5_DPI_R3_G3,5,3,G3,iVar,INH_5_DPI_R3_G3.vcf
2,VEEV_INH,38,A,G,None,2530,0.004348,None,None,None,...,None,None,None,None,INH_5_DPI_R3_G3,5,3,G3,iVar,INH_5_DPI_R3_G3.vcf
3,VEEV_INH,53,A,G,None,2688,0.001116,None,None,synonymous|NSP1|NSP1_transcript|protein_coding...,...,protein_coding,+,3K,53A>G,INH_5_DPI_R3_G3,5,3,G3,iVar,INH_5_DPI_R3_G3.vcf
4,VEEV_INH,74,A,G,None,2941,0.001360,None,None,synonymous|NSP1|NSP1_transcript|protein_coding...,...,protein_coding,+,10E,74A>G,INH_5_DPI_R3_G3,5,3,G3,iVar,INH_5_DPI_R3_G3.vcf


# Filter

I applied a 1000 depth filter instead of 5000 on the first run, so I'll correct here

In [43]:
# Filter by coverage >5000 and frequency >0.1%
print("FILTERING BY COVERAGE >5000 AND FREQUENCY >0.1%")
print("="*50)

# Before filtering
print(f"LoFreq variants before filtering: {len(lofreq_df)}")
print(f"iVar variants before filtering: {len(ivar_df)}")

# Apply filters
lofreq_filtered = lofreq_df[(lofreq_df['DP'] > 5000) & (lofreq_df['AF'] > 0.001)]
ivar_filtered = ivar_df[(ivar_df['DP'] > 5000) & (ivar_df['AF'] > 0.001)]

# After filtering
print(f"\nAfter coverage and frequency filtering:")
print(f"LoFreq variants: {len(lofreq_filtered)}")
print(f"iVar variants: {len(ivar_filtered)}")

# Show how many were filtered out
print(f"\nFiltered out:")
print(f"LoFreq: {len(lofreq_df) - len(lofreq_filtered)} variants removed")
print(f"iVar: {len(ivar_df) - len(ivar_filtered)} variants removed")

# Update the dataframes
lofreq_df = lofreq_filtered
ivar_df = ivar_filtered

print(f"\nTotal variants remaining: {len(lofreq_df) + len(ivar_df)}")
print(f"All variants now have >5000x coverage and >0.1% frequency")

FILTERING BY COVERAGE >5000 AND FREQUENCY >0.1%
LoFreq variants before filtering: 3220
iVar variants before filtering: 3954

After coverage and frequency filtering:
LoFreq variants: 1319
iVar variants: 2777

Filtered out:
LoFreq: 1901 variants removed
iVar: 1177 variants removed

Total variants remaining: 4096
All variants now have >5000x coverage and >0.1% frequency


# Count Nonsynonymous mutations

In [44]:
# Filter for missense variants
lofreq_nonsyn = lofreq_df[lofreq_df['consequence'] == 'missense']
ivar_nonsyn = ivar_df[ivar_df['consequence'] == '*missense']

# Create summary tables
lofreq_table = lofreq_nonsyn.groupby(['dpi', 'replicate']).size().reset_index(name='LoFreq')
ivar_table = ivar_nonsyn.groupby(['dpi', 'replicate']).size().reset_index(name='iVar')

# Create complete index for all DPI-replicate combinations
dpi_values = [1, 3, 5]
replicate_values = [1, 2, 3, 4]

# Create all combinations
import itertools
all_combinations = pd.DataFrame(list(itertools.product(dpi_values, replicate_values)), 
                               columns=['dpi', 'replicate'])

# Merge with actual data
result = all_combinations.merge(lofreq_table, on=['dpi', 'replicate'], how='left')
result = result.merge(ivar_table, on=['dpi', 'replicate'], how='left')

# Fill NaN with 0
result['LoFreq'] = result['LoFreq'].fillna(0).astype(int)
result['iVar'] = result['iVar'].fillna(0).astype(int)

# Sort by DPI and replicate
result = result.sort_values(['dpi', 'replicate'])

# Display as nice HTML table in notebook
result

,dpi,replicate,LoFreq,iVar
0,1,1,0,0
1,1,2,0,0
2,1,3,0,0
3,1,4,0,0
4,3,1,29,0
5,3,2,67,0
6,3,3,96,0
7,3,4,89,0
8,5,1,89,0
9,5,2,93,0


# Mutations per Gene

In [45]:
# Mutations per Gene - Day 3 and Day 5
print("MUTATIONS PER GENE BY DPI")
print("="*40)

# Combine both datasets for analysis
all_variants = []
if not lofreq_df.empty:
    all_variants.append(lofreq_df)
if not ivar_df.empty:
    all_variants.append(ivar_df)

if all_variants:
    combined_df = pd.concat(all_variants, ignore_index=True)
    
    # Day 3 mutations per gene
    print("\nDAY 3 MUTATIONS:")
    day3_data = combined_df[combined_df['dpi'] == 3]
    
    lofreq_day3 = day3_data[day3_data['variant_caller'] == 'LoFreq'].groupby('gene').size().reset_index(name='LoFreq')
    ivar_day3 = day3_data[day3_data['variant_caller'] == 'iVar'].groupby('gene').size().reset_index(name='iVar')
    
    # Get all genes present in either dataset
    all_genes_day3 = set()
    if not lofreq_day3.empty:
        all_genes_day3.update(lofreq_day3['gene'].dropna())
    if not ivar_day3.empty:
        all_genes_day3.update(ivar_day3['gene'].dropna())
    
    if all_genes_day3:
        genes_day3_df = pd.DataFrame({'gene': sorted(list(all_genes_day3))})
        day3_result = genes_day3_df.merge(lofreq_day3, on='gene', how='left')
        day3_result = day3_result.merge(ivar_day3, on='gene', how='left')
        day3_result['LoFreq'] = day3_result['LoFreq'].fillna(0).astype(int)
        day3_result['iVar'] = day3_result['iVar'].fillna(0).astype(int)
        display(day3_result)
    else:
        print("No genes found for Day 3")
    
    # Day 5 mutations per gene
    print("\nDAY 5 MUTATIONS:")
    day5_data = combined_df[combined_df['dpi'] == 5]
    
    lofreq_day5 = day5_data[day5_data['variant_caller'] == 'LoFreq'].groupby('gene').size().reset_index(name='LoFreq')
    ivar_day5 = day5_data[day5_data['variant_caller'] == 'iVar'].groupby('gene').size().reset_index(name='iVar')
    
    # Get all genes present in either dataset
    all_genes_day5 = set()
    if not lofreq_day5.empty:
        all_genes_day5.update(lofreq_day5['gene'].dropna())
    if not ivar_day5.empty:
        all_genes_day5.update(ivar_day5['gene'].dropna())
    
    if all_genes_day5:
        genes_day5_df = pd.DataFrame({'gene': sorted(list(all_genes_day5))})
        day5_result = genes_day5_df.merge(lofreq_day5, on='gene', how='left')
        day5_result = day5_result.merge(ivar_day5, on='gene', how='left')
        day5_result['LoFreq'] = day5_result['LoFreq'].fillna(0).astype(int)
        day5_result['iVar'] = day5_result['iVar'].fillna(0).astype(int)
        display(day5_result)
    else:
        print("No genes found for Day 5")
else:
    print("No variant data available")

MUTATIONS PER GENE BY DPI

DAY 3 MUTATIONS:


/tmp/ipykernel_2952986/4026733524.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(all_variants, ignore_index=True)


,gene,LoFreq,iVar
0,6k,7,86
1,Capsid,29,82
2,E1,83,173
3,E2,71,129
4,E3,0,7
5,NSP1,50,125
6,NSP2,100,202
7,NSP3,84,157
8,NSP4,79,185



DAY 5 MUTATIONS:


,gene,LoFreq,iVar
0,6k,12,93
1,Capsid,36,93
2,E1,93,187
3,E2,69,137
4,E3,2,9
5,NSP1,68,171
6,NSP2,124,317
7,NSP3,128,266
8,NSP4,128,301


# Unique Non-Syn Mutations

In [46]:
# Filter for missense variants found in ALL 4 replicates
print("MISSENSE MUTATIONS FOUND IN ALL 4 REPLICATES")
print("="*50)

# Filter for missense variants
lofreq_missense = lofreq_df[lofreq_df['consequence'] == 'missense']
ivar_missense = ivar_df[ivar_df['consequence'] == '*missense']

# Create detailed mutation names
lofreq_missense['mutation'] = lofreq_missense['gene'] + ':p.' + lofreq_missense['amino_acid_change']
ivar_missense['mutation'] = ivar_missense['gene'] + ':p.' + ivar_missense['amino_acid_change']

# LoFreq table - mutations in all 4 replicates
print("\nLOFREQ MUTATIONS IN ALL 4 REPLICATES:")

# Day 3 - find mutations present in all 4 replicates
lofreq_day3_all = lofreq_missense[lofreq_missense['dpi'] == 3]
lofreq_day3_counts = lofreq_day3_all.groupby('mutation')['replicate'].nunique()
lofreq_day3_consensus = sorted(lofreq_day3_counts[lofreq_day3_counts == 4].index)

# Day 5 - find mutations present in all 4 replicates  
lofreq_day5_all = lofreq_missense[lofreq_missense['dpi'] == 5]
lofreq_day5_counts = lofreq_day5_all.groupby('mutation')['replicate'].nunique()
lofreq_day5_consensus = sorted(lofreq_day5_counts[lofreq_day5_counts == 4].index)

# Create LoFreq DataFrame
from itertools import zip_longest
lofreq_data = list(zip_longest(lofreq_day3_consensus, lofreq_day5_consensus, fillvalue=''))
lofreq_df_result = pd.DataFrame(lofreq_data, columns=['Day 3 (4/4 replicates)', 'Day 5 (4/4 replicates)'])
display(lofreq_df_result)

# iVar table - mutations in all 4 replicates
print("\nIVAR MUTATIONS IN ALL 4 REPLICATES:")

# Day 3 - find mutations present in all 4 replicates
ivar_day3_all = ivar_missense[ivar_missense['dpi'] == 3]
ivar_day3_counts = ivar_day3_all.groupby('mutation')['replicate'].nunique()
ivar_day3_consensus = sorted(ivar_day3_counts[ivar_day3_counts == 4].index)

# Day 5 - find mutations present in all 4 replicates
ivar_day5_all = ivar_missense[ivar_missense['dpi'] == 5]
ivar_day5_counts = ivar_day5_all.groupby('mutation')['replicate'].nunique()
ivar_day5_consensus = sorted(ivar_day5_counts[ivar_day5_counts == 4].index)

# Create iVar DataFrame
ivar_data = list(zip_longest(ivar_day3_consensus, ivar_day5_consensus, fillvalue=''))
ivar_df_result = pd.DataFrame(ivar_data, columns=['Day 3 (4/4 replicates)', 'Day 5 (4/4 replicates)'])
display(ivar_df_result)

print(f"\nLoFreq consensus mutations: {len(lofreq_day3_consensus)} Day 3, {len(lofreq_day5_consensus)} Day 5")
print(f"iVar consensus mutations: {len(ivar_day3_consensus)} Day 3, {len(ivar_day5_consensus)} Day 5")

MISSENSE MUTATIONS FOUND IN ALL 4 REPLICATES

LOFREQ MUTATIONS IN ALL 4 REPLICATES:


/tmp/ipykernel_2952986/1466495245.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lofreq_missense['mutation'] = lofreq_missense['gene'] + ':p.' + lofreq_missense['amino_acid_change']


,Day 3 (4/4 replicates),Day 5 (4/4 replicates)
0,6k:p.4W>4R,6k:p.27A>27T
1,E1:p.1Y>1C,6k:p.4W>4R
2,E1:p.46Y>46C,E1:p.1Y>1C
3,,E1:p.46Y>46C
4,,NSP2:p.221V>221A
5,,NSP2:p.395K>395R
6,,NSP2:p.526L>526P
7,,NSP2:p.788R>788G
8,,NSP3:p.158R>158G
9,,NSP3:p.162V>162A



IVAR MUTATIONS IN ALL 4 REPLICATES:


,Day 3 (4/4 replicates),Day 5 (4/4 replicates)



LoFreq consensus mutations: 3 Day 3, 20 Day 5
iVar consensus mutations: 0 Day 3, 0 Day 5


# mutations at freq >1%

In [47]:
# Mutations with frequency >1% (coding regions only)
print("MUTATIONS WITH FREQUENCY >1% (CODING REGIONS ONLY)")
print("="*50)

# Filter for mutations >1% frequency AND with valid gene/amino acid info
lofreq_high_freq = lofreq_df[(lofreq_df['AF'] > 0.01) & 
                             (lofreq_df['gene'].notna()) & 
                             (lofreq_df['amino_acid_change'].notna())]

ivar_high_freq = ivar_df[(ivar_df['AF'] > 0.01) & 
                         (ivar_df['gene'].notna()) & 
                         (ivar_df['amino_acid_change'].notna())]

# LoFreq table
print("\nLOFREQ MUTATIONS >1% (CODING):")
if not lofreq_high_freq.empty:
    lofreq_result = lofreq_high_freq[['dpi', 'gene', 'amino_acid_change', 'AF']].copy()
    lofreq_result['NS Mutation'] = lofreq_result['gene'] + ':p.' + lofreq_result['amino_acid_change']
    lofreq_result['Freq %'] = (lofreq_result['AF'] * 100).round(1)
    lofreq_table = lofreq_result[['dpi', 'NS Mutation', 'Freq %']].rename(columns={'dpi': 'DPI'})
    lofreq_table = lofreq_table.sort_values(['DPI', 'Freq %'], ascending=[True, False])
    display(lofreq_table)
else:
    print("No LoFreq coding mutations >1%")

# iVar table
print("\nIVAR MUTATIONS >1% (CODING):")
if not ivar_high_freq.empty:
    ivar_result = ivar_high_freq[['dpi', 'gene', 'amino_acid_change', 'AF']].copy()
    ivar_result['NS Mutation'] = ivar_result['gene'] + ':p.' + ivar_result['amino_acid_change']
    ivar_result['Freq %'] = (ivar_result['AF'] * 100).round(1)
    ivar_table = ivar_result[['dpi', 'NS Mutation', 'Freq %']].rename(columns={'dpi': 'DPI'})
    ivar_table = ivar_table.sort_values(['DPI', 'Freq %'], ascending=[True, False])
    display(ivar_table)
else:
    print("No iVar coding mutations >1%")

print(f"\nLoFreq coding mutations >1%: {len(lofreq_high_freq)}")
print(f"iVar coding mutations >1%: {len(ivar_high_freq)}")

MUTATIONS WITH FREQUENCY >1% (CODING REGIONS ONLY)

LOFREQ MUTATIONS >1% (CODING):


,DPI,NS Mutation,Freq %
1201,3,NSP2:p.686K>686I,91.4
1689,3,E2:p.402L,67.9
1720,3,E1:p.361I,27.7
897,3,NSP3:p.158R>158G,3.3
1215,3,NSP3:p.158R>158G,3.1
443,3,NSP3:p.158R>158G,2.9
930,3,NSP4:p.318A>318V,2.4
431,3,NSP2:p.686K>686I,2.1
496,3,NSP4:p.318A>318V,2.0
424,3,NSP2:p.626V>626A,1.2



IVAR MUTATIONS >1% (CODING):


,DPI,NS Mutation,Freq %
1638,3,NSP2:p.686K>686I,96.6
2479,3,E2:p.402L,70.1
2533,3,E1:p.361I,28.3
1288,3,NSP3:p.158R>158G,3.5
1661,3,NSP3:p.158R>158G,3.3
2498,3,6k:p.48V>48G,3.3
765,3,NSP3:p.158R>158G,3.2
1834,3,6k:p.48V>48G,3.2
948,3,6k:p.48V>48G,3.0
1375,3,NSP4:p.318A>318V,2.5



LoFreq coding mutations >1%: 29
iVar coding mutations >1%: 54
